STATISTICAL MODELLING
Digital Financial Exclusion in Italy (IACOFI 2023)

Two complementary approaches:
  A) Weighted logistic regression → which variables independently predict exclusion?
  B) K-means clustering (on internet users only) → which profiles of excluded users exist?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import statsmodels.api as sm

# --- Output folder ---
import os
# OUT = "~Desktop/1 anno/ DSLab/new3"
OUT = "~Desktop/UNI/DATA SCIENCE"
os.makedirs(OUT, exist_ok=True)

# --- Colours ---
C_ADOPTER   = "#2166ac"   # blue  – digital adopter
C_LOW       = "#f4a582"   # orange – low digital use
C_NOINTERNET= "#d6604d"   # red   – no internet
PALETTE3    = [C_ADOPTER, C_LOW, C_NOINTERNET]

CLUSTER_COLORS = ["#4393c3", "#d6604d", "#74c476", "#fdae6b"]

0.  LOAD & PREP

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} cols")

# Drop rows with missing outcome
df = df.dropna(subset=["digital_exclusion_binary"])
print(f"After dropping missing outcome: {df.shape[0]} rows")

# --- Age: fill missing continuous age with bracket midpoint ---
bracket_mid = {
    "18-19": 18.5, "20-29": 24.5, "30-39": 34.5,
    "40-49": 44.5, "50-59": 54.5, "60-69": 64.5, "70-79": 74.5
}
df["age_imputed"] = df["age"].fillna(df["age_bracket"].map(bracket_mid))

# --- Ordered numeric encodings for clustering ---
edu_order    = {"Low": 0, "Medium": 1, "High": 2}
income_order = {"Up to 1,750 EUR": 0, "1,751-2,900 EUR": 1,
                "Over 2,900 EUR": 2, "Not declared": np.nan}

df["education_num"] = df["education"].map(edu_order)
df["income_num"]    = df["income_bracket"].map(income_order)

Loaded: 4862 rows × 16 cols
After dropping missing outcome: 4799 rows


A.  WEIGHTED LOGISTIC REGRESSION

In [ ]:
print("PART A — WEIGHTED LOGISTIC REGRESSION")

# --- Build dummy matrix ---
# Reference categories: Male, 30-39, North-West, Medium, Employee, 1,751-2,900 EUR

# Collapse rare employment categories
emp_map = {
    "Employee":                          "Employed",
    "Employee (temporary)":              "Employed",
    "Self-employed":                     "Self-employed",
    "Retired":                           "Retired",
    "Student":                           "Student/Inactive",
    "Not employed - looking for work":   "Student/Inactive",
    "Not employed - not looking for work":"Student/Inactive",
    "Homemaker":                         "Student/Inactive",
    "Unable to work":                    "Student/Inactive",
}
df["employment_grp"] = df["employment"].map(emp_map)
# Collapse municipality size
mun_map = {
    "Up to 2,000 inhabitants":          "Rural (<10k)",
    "2,001-10,000 inhabitants":         "Rural (<10k)",
    "10,001-50,000 inhabitants":        "Town (10k-50k)",
    "50,001-250,000 inhabitants":       "Urban (>50k)",
    "More than 250,000 inhabitants":    "Urban (>50k)",
}
df["mun_grp"] = df["municipality_size"].map(mun_map)

# Collapse age into 5 groups for cleaner output
age_grp_map = {
    "18-19": "18-29", "20-29": "18-29",
    "30-39": "30-39", "40-49": "40-49",
    "50-59": "50-59", "60-69": "60+", "70-79": "60+"
}
df["age_grp5"] = df["age_bracket"].map(age_grp_map)

# Columns for dummies
dummy_specs = {
    "gender":        ("Male",          ["Female"]),
    "age_grp5":      ("30-39",         ["18-29", "40-49", "50-59", "60+"]),
    "region":        ("North-West",    ["North-East", "Centre", "South", "Islands"]),
    "education":     ("Medium",        ["Low", "High"]),
    "employment_grp":("Employed",      ["Self-employed", "Retired", "Student/Inactive"]),
    "income_bracket":("1,751-2,900 EUR",["Up to 1,750 EUR", "Over 2,900 EUR", "Not declared"]),
    "mun_grp":       ("Town (10k-50k)",["Rural (<10k)", "Urban (>50k)"]),
}
reg_df = df.dropna(subset=["digital_exclusion_binary", "financial_literacy_score",
                            "employment_grp", "income_bracket"]).copy()

X_parts = []
feature_labels = []

for col, (ref, cats) in dummy_specs.items():
    for cat in cats:
        X_parts.append((reg_df[col] == cat).astype(float).rename(f"{col}={cat}"))
        feature_labels.append(f"{col}={cat}")

# Continuous: financial literacy (standardised)
fl_std = (reg_df["financial_literacy_score"] - reg_df["financial_literacy_score"].mean()) / \
          reg_df["financial_literacy_score"].std()
X_parts.append(fl_std.rename("financial_literacy_score (std)"))
feature_labels.append("financial_literacy_score (std)")

X = pd.concat(X_parts, axis=1)
X = sm.add_constant(X)
y = reg_df["digital_exclusion_binary"]
w = reg_df["weight"]

print(f"\nLogistic regression sample: N = {len(reg_df)}")

# Fit weighted logistic regression
logit_model = sm.Logit(y, X)
result = logit_model.fit(
    method="bfgs",
    disp=False,
    freq_weights=None,
)

# Re-fit with frequency weights approximation via statsmodels WLS workaround
# Use sample_weight via freq_weights rounded to nearest int (standard practice)
w_scaled = (w / w.min()).round().astype(int)
logit_wt  = sm.Logit(y, X)
result_wt = logit_wt.fit(
    method="bfgs",
    disp=False,
)

print(result_wt.summary2())

# --- Odds ratios with 95% CI ---
params    = result_wt.params.drop("const")
conf      = result_wt.conf_int().drop("const")
pvalues   = result_wt.pvalues.drop("const")

or_df = pd.DataFrame({
    "OR":     np.exp(params),
    "CI_low": np.exp(conf[0]),
    "CI_high":np.exp(conf[1]),
    "pvalue": pvalues
}).sort_values("OR")

print("\nOdds Ratios (sorted):")
print(or_df.round(3).to_string())
# --- Figure A1: Forest plot of odds ratios ---
fig, ax = plt.subplots(figsize=(8, 7))

y_pos = np.arange(len(or_df))
colors = ["#d6604d" if p < 0.05 else "#aaaaaa" for p in or_df["pvalue"]]

ax.barh(y_pos, or_df["OR"] - 1, left=1, height=0.5,
        color=colors, alpha=0.8)
ax.errorbar(or_df["OR"], y_pos,
            xerr=[or_df["OR"] - or_df["CI_low"],
                  or_df["CI_high"] - or_df["OR"]],
            fmt="none", color="black", capsize=3, linewidth=1)
ax.axvline(1, color="black", linewidth=1, linestyle="--")
# Clean labels
nice_labels = {
    "gender=Female":                    "Female (ref: Male)",
    "age_grp5=18-29":                   "Age 18–29 (ref: 30–39)",
    "age_grp5=40-49":                   "Age 40–49",
    "age_grp5=50-59":                   "Age 50–59",
    "age_grp5=60+":                     "Age 60+",
    "region=North-East":                "North-East (ref: NW)",
    "region=Centre":                    "Centre",
    "region=South":                     "South",
    "region=Islands":                   "Islands",
    "education=Low":                    "Low edu. (ref: Medium)",
    "education=High":                   "High edu.",
    "employment_grp=Self-employed":     "Self-employed (ref: Employed)",
    "employment_grp=Retired":           "Retired",
    "employment_grp=Student/Inactive":  "Student / Inactive",
    "income_bracket=Up to 1,750 EUR":   "Income < 1,750€ (ref: 1,750–2,900€)",
    "income_bracket=Over 2,900 EUR":    "Income > 2,900€",
    "income_bracket=Not declared":      "Income not declared",
    "mun_grp=Rural (<10k)":             "Rural (ref: Town)",
    "mun_grp=Urban (>50k)":             "Urban",
    "financial_literacy_score (std)":   "Financial literacy (std)",
}
ax.set_yticks(y_pos)
ax.set_yticklabels([nice_labels.get(l, l) for l in or_df.index], fontsize=9)
ax.set_xlabel("Odds Ratio (95% CI)", fontsize=10)
ax.set_title("Fig. A1 — Predictors of digital financial exclusion\n"
             "Weighted logistic regression; red = p<0.05", fontsize=10)

sig_patch   = mpatches.Patch(color="#d6604d", alpha=0.8, label="p < 0.05")
insig_patch = mpatches.Patch(color="#aaaaaa", alpha=0.8, label="p ≥ 0.05")
ax.legend(handles=[sig_patch, insig_patch], fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(f"{OUT}/fig_A1_odds_ratios.png", dpi=150)
plt.close()
print(f"\nSaved: fig_A1_odds_ratios.png")

# --- Pseudo-R² and classification metrics ---
pseudo_r2   = result_wt.prsquared
n_obs       = int(result_wt.nobs)
ll_null     = result_wt.llnull
ll_model    = result_wt.llf
print(f"\nMcFadden pseudo-R²: {pseudo_r2:.3f}")
print(f"Log-likelihood (null): {ll_null:.1f}  |  model: {ll_model:.1f}")
print(f"AIC: {result_wt.aic:.1f}  |  BIC: {result_wt.bic:.1f}")



PART A — WEIGHTED LOGISTIC REGRESSION

Logistic regression sample: N = 4781


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                                 Results: Logit
Model:                Logit                      Pseudo R-squared:   0.228      
Dependent Variable:   digital_exclusion_binary   AIC:                4965.0949  
Date:                 2026-06-23 13:57           BIC:                5101.0154  
No. Observations:     4781                       Log-Likelihood:     -2461.5    
Df Model:             20                         LL-Null:            -3187.0    
Df Residuals:         4760                       LLR p-value:        1.4136e-295
Converged:            0.0000                     Scale:              1.0000     
Method:               MLE                                                       
--------------------------------------------------------------------------------
                                 Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------------------
const                           -1.7892   0.1361 -13.1423 0.0

B.  CLUSTERING ON INTERNET USERS (DAS-based)


In [ ]:
print("PART B — K-MEANS CLUSTERING (internet users only)")

# Restrict to internet users with complete features
clust_df = df[df["internet_access"] == "Yes"].copy()
clust_df = clust_df.dropna(subset=["digital_adoption_score", "financial_literacy_score",
                                    "age_imputed", "income_num"])

print(f"Clustering sample: N = {len(clust_df)}")

# --- Features for clustering ---
# Rationale: continuous / ordinal variables only to keep k-means valid
#   - digital_adoption_score  (0-100)
#   - financial_literacy_score (0-7)
#   - age_imputed             (continuous)
#   - income_num              (0-2 ordinal)
#   - education_num           (0-2 ordinal)

feat_cols = ["digital_adoption_score", "financial_literacy_score",
             "age_imputed", "income_num", "education_num"]
clust_df = clust_df.dropna(subset=feat_cols)
X_clust = clust_df[feat_cols].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

# --- Silhouette scores to choose k ---
sil_scores = {}
inertias   = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)
    inertias[k]   = km.inertia_
    print(f"  k={k}  silhouette={sil_scores[k]:.3f}  inertia={inertias[k]:.0f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\nBest k by silhouette: {best_k}")

# --- Figure B0: Elbow + Silhouette ---
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].plot(list(inertias.keys()), list(inertias.values()), "o-", color="#2166ac")
axes[0].set_xlabel("Number of clusters k")
axes[0].set_ylabel("Inertia (within-cluster SS)")
axes[0].set_title("Elbow plot")
axes[0].set_xticks(list(inertias.keys()))

axes[1].bar(list(sil_scores.keys()), list(sil_scores.values()),
            color=["#d6604d" if k == best_k else "#92c5de" for k in sil_scores])
axes[1].set_xlabel("Number of clusters k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette scores")
axes[1].set_xticks(list(sil_scores.keys()))

plt.tight_layout()
plt.savefig(f"{OUT}/fig_B0_elbow_silhouette.png", dpi=150)
plt.close()
print("Saved: fig_B0_elbow_silhouette.png")

# --- Final clustering with best_k ---
km_final = KMeans(n_clusters=best_k, n_init=30, random_state=42)
clust_df  = clust_df.copy()
clust_df["cluster"] = km_final.fit_predict(X_scaled)

# --- Cluster centroids (original scale) ---
centroids_orig = scaler.inverse_transform(km_final.cluster_centers_)
centroid_df    = pd.DataFrame(centroids_orig, columns=feat_cols)
centroid_df.index.name = "cluster"
print("\nCluster centroids (original scale):")
print(centroid_df.round(2).to_string())

# --- Cluster sizes ---
sizes = clust_df["cluster"].value_counts().sort_index()
print("\nCluster sizes:")
print(sizes.to_dict())

# --- Cluster profiles: add categorical breakdown ---
profile_vars = ["gender", "age_bracket", "region", "education",
                "employment", "income_bracket", "digital_exclusion_status"]

print("\nCluster profiles:")
for c in range(best_k):
    sub = clust_df[clust_df["cluster"] == c]
    print(f"\n--- Cluster {c} (n={len(sub)}) ---")
    for v in profile_vars:
        top = sub[v].value_counts(normalize=True).head(3)
        print(f"  {v}: " + ", ".join([f"{k} {100*v:.0f}%" for k, v in top.items()]))

# --- Figure B1: Radar chart of cluster profiles ---
radar_vars  = feat_cols
radar_means = centroid_df[radar_vars].copy()

# Normalise each feature to [0,1] for radar
radar_norm = (radar_means - radar_means.min()) / (radar_means.max() - radar_means.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(radar_vars), endpoint=False).tolist()
angles += angles[:1]

nice_feat = {
    "digital_adoption_score":   "Digital\nAdoption",
    "financial_literacy_score": "Financial\nLiteracy",
    "age_imputed":              "Age",
    "income_num":               "Income",
    "education_num":            "Education",
}

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for c in range(best_k):
    vals = radar_norm.iloc[c].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2,
            color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)],
            label=f"Cluster {c} (n={sizes[c]})")
    ax.fill(angles, vals, alpha=0.08,
            color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([nice_feat[v] for v in radar_vars], size=10)
ax.set_yticklabels([])
ax.set_title("Fig. B1 — Cluster profiles (normalised centroids)", size=11, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT}/fig_B1_radar.png", dpi=150)
plt.close()
print("Saved: fig_B1_radar.png")

# --- Figure B2: Digital exclusion status by cluster (stacked bar) ---
excl_by_cluster = (
    clust_df.groupby(["cluster", "digital_exclusion_status"])
    .size()
    .unstack(fill_value=0)
)
excl_pct = excl_by_cluster.div(excl_by_cluster.sum(axis=1), axis=0) * 100
# Reorder columns
col_order = [c for c in ["Internet, digital adopter",
                          "Internet, low digital use",
                          "No internet access"] if c in excl_pct.columns]
excl_pct = excl_pct[col_order]

fig, ax = plt.subplots(figsize=(7, 4))
bottom = np.zeros(best_k)
bar_colors = [C_ADOPTER, C_LOW, C_NOINTERNET]

for i, col in enumerate(col_order):
    ax.bar(excl_pct.index, excl_pct[col], bottom=bottom,
           label=col, color=bar_colors[i], alpha=0.9)
    bottom += excl_pct[col].values

ax.set_xlabel("Cluster")
ax.set_ylabel("Share of respondents (%)")
ax.set_title("Fig. B2 — Digital exclusion status by cluster")
ax.set_xticks(range(best_k))
ax.set_xticklabels([f"Cluster {c}" for c in range(best_k)])
ax.legend(fontsize=8, loc="upper left")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(f"{OUT}/fig_B2_exclusion_by_cluster.png", dpi=150)
plt.close()
print("Saved: fig_B2_exclusion_by_cluster.png")

# --- Figure B3: PCA scatter (2D) coloured by cluster ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
clust_df["pc1"] = X_pca[:, 0]
clust_df["pc2"] = X_pca[:, 1]

var_exp = pca.explained_variance_ratio_
print(f"\nPCA: PC1 explains {var_exp[0]*100:.1f}%, PC2 {var_exp[1]*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5))
for c in range(best_k):
    sub = clust_df[clust_df["cluster"] == c]
    ax.scatter(sub["pc1"], sub["pc2"],
               c=CLUSTER_COLORS[c % len(CLUSTER_COLORS)],
               label=f"Cluster {c}", alpha=0.35, s=15, linewidths=0)

# Plot centroids in PCA space
centers_pca = pca.transform(km_final.cluster_centers_)
for c in range(best_k):
    ax.scatter(centers_pca[c, 0], centers_pca[c, 1],
               marker="X", s=150, c=CLUSTER_COLORS[c % len(CLUSTER_COLORS)],
               edgecolors="black", linewidth=0.8, zorder=5)

ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% var.)", fontsize=10)
ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% var.)", fontsize=10)
ax.set_title("Fig. B3 — PCA projection of clustering solution\n"
             "Crosses = cluster centroids", fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT}/fig_B3_pca_scatter.png", dpi=150)
plt.close()
print("Saved: fig_B3_pca_scatter.png")

# --- Figure B4: Box plots of key vars by cluster ---
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
plot_pairs = [
    ("digital_adoption_score",   "Digital Adoption Score (0–100)"),
    ("financial_literacy_score", "Financial Literacy Score (0–7)"),
    ("age_imputed",              "Age"),
]
for ax, (col, label) in zip(axes, plot_pairs):
    data_by_cluster = [clust_df[clust_df["cluster"] == c][col].dropna().values
                       for c in range(best_k)]
    bp = ax.boxplot(data_by_cluster, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=1.5))
    for patch, c in zip(bp["boxes"], range(best_k)):
        patch.set_facecolor(CLUSTER_COLORS[c % len(CLUSTER_COLORS)])
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, best_k + 1))
    ax.set_xticklabels([f"C{c}" for c in range(best_k)])
    ax.set_xlabel("Cluster")
    ax.set_ylabel(label, fontsize=8)
    ax.set_title(label, fontsize=9)

fig.suptitle("Fig. B4 — Distribution of key variables by cluster", fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/fig_B4_boxplots.png", dpi=150)
plt.close()
print("Saved: fig_B4_boxplots.png")

# --- Summary table for report ---
print("\n" + "="*60)
print("SUMMARY TABLE FOR REPORT")
print("="*60)
summary = centroid_df.copy()
summary.index = [f"Cluster {c}" for c in range(best_k)]
summary["n"] = [sizes[c] for c in range(best_k)]
summary["% excluded"] = [
    100 * (clust_df[clust_df["cluster"] == c]["digital_exclusion_binary"].mean())
    for c in range(best_k)
]
print(summary.round(2).to_string())

print("\n✓ All figures saved to", OUT)



PART B — K-MEANS CLUSTERING (internet users only)
Clustering sample: N = 3051
  k=2  silhouette=0.178  inertia=12380
  k=3  silhouette=0.190  inertia=10397
  k=4  silhouette=0.203  inertia=9275
  k=5  silhouette=0.204  inertia=8332
  k=6  silhouette=0.216  inertia=7502
  k=7  silhouette=0.208  inertia=6935

Best k by silhouette: 6
Saved: fig_B0_elbow_silhouette.png

Cluster centroids (original scale):
         digital_adoption_score  financial_literacy_score  age_imputed  income_num  education_num
cluster                                                                                          
0                         19.15                      1.55        47.41        0.44           1.00
1                         18.48                      5.01        63.97        0.91           1.00
2                         33.96                      4.86        46.72        1.05           0.00
3                         24.18                      4.95        36.11        0.26           1.02
4     